# 📊 Avaliação de Modelo Fine-Tuned com LoRA

Este notebook avalia um modelo de linguagem fine-tuned usando as principais métricas de NLP. O dataset utilizado é um conjunto de Q&A automotivo (`dataset.jsonl`) e o modelo base é o `distilgpt2` com adaptadores LoRA.

---

## 🗺️ Métricas Avaliadas

| Métrica | Foco | Intervalo Típico |
|---|---|---|
| **Perplexidade (PPL)** | Capacidade preditiva do modelo | Menor = melhor |
| **BLEU** | Precisão de n-gramas vs. referência | 0–100 (maior = melhor) |
| **ROUGE** | Recall de sobreposição de conteúdo | 0–1 (maior = melhor) |
| **Fidelidade (Faithfulness)** | Se a resposta é fiel ao contexto | 0–1 (maior = melhor) |
| **Relevância da Resposta** | Se a resposta é útil à pergunta | 0–1 (maior = melhor) |
| **Aderência ao Plano** | Se o agente segue o plano de ação | 0–1 (maior = melhor) |

---

## 📦 Célula 1 — Instalação de Dependências

Instalamos todas as bibliotecas necessárias. Execute uma única vez.

In [1]:
!pip install -q transformers peft torch sacrebleu rouge-score nltk pandas matplotlib seaborn tqdm

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ Todas as dependências instaladas com sucesso!')

✅ Todas as dependências instaladas com sucesso!


## 🔌 Célula 2 — Importações

Importamos todas as bibliotecas que serão usadas ao longo do notebook.

In [2]:
# ============================================================
#  IMPORTAÇÕES
# ============================================================

import json
import math
import re
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm import tqdm

# Transformers & PEFT
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Métricas NLP
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize

warnings.filterwarnings('ignore')

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ Importações concluídas!')
print(f'   PyTorch: {torch.__version__}')
print(f'   Dispositivo disponível: {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

/home/henderson/Documentos/rag/Pipeline-RAG-com-Fine-Tuning-LoRA-e-Disponibiliza-o-via-API-RESTful/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Importações concluídas!
   PyTorch: 2.12.0+cu130
   Dispositivo disponível: CPU


## 📂 Célula 3 — Carregamento do Dataset

Lemos o arquivo `dataset.jsonl`. Cada linha contém:
- **`instruction`**: o que o usuário quer saber
- **`input`**: contexto adicional (veículo, modelo, etc.)
- **`output`**: resposta esperada (ground truth)

In [3]:
# ============================================================
#  CARREGAMENTO DO DATASET
# ============================================================
# Ajuste o caminho caso o arquivo esteja em outro diretório.

DATASET_PATH = 'dataset.jsonl'

samples = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f'✅ Dataset carregado com {len(samples)} amostras.\n')

# Exibição amigável das primeiras amostras
df_raw = pd.DataFrame(samples)
df_raw[['instruction', 'input', 'output']].head(3).style \
    .set_caption('📋 Primeiras amostras do dataset') \
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

✅ Dataset carregado com 106 amostras.



,instruction,input,output
0,Quais são as principais áreas de atuação do Ministério da Saúde e como elas contribuem para a gestão da emergência sanitária de 2021?,"DESEMPENHO DA GESTÃO 29 Ministério em números 30 2.1 Atenção Primária 32 2.2 Atenção Especializada 68 2.3 Assistência farmacêutica 141 2.4 Vigilância em Saúde 148 2.5 Ciência, Tecnologia, Inovação e Insumos Estratégicos 174 2.6 Saúde Indígena 187 2.7 Aprimoramento da Gestão e Educação na Área da Saúde 194 2.8 Saúde Digital 199 Em 2021, o país teve como desafio, pelo segundo ano, a maior emergência sanitária já enfrentada. O Ministério da Saúde, em conjunto com os Esta - dos, os Municípios e o Distrito Federal, não mediu esforços para execução das ações necessárias para enfrentamento da covid-19, visando conter a sua proliferação e garantir o atendimento necessário aos pacientes con - taminados. Destacam-se as transferências financeiras aos demais entes federativos visando a abertura e o","As principais áreas de atuação do Ministério da Saúde incluem Atenção Primária, Atenção Especializada, Assistência Farmacêutica, Vigilância em Saúde, Ciência, Tecnologia, Inovação e Insumos Estratégicos, Saúde Indígena, Aprimoramento da Gestão e Educação na Área da Saúde e Saúde Digital. Essas áreas contribuem para a gestão da emergência sanitária de 2021 ao fornecer estruturas de atendimento, acesso a medicamentos, monitoramento de doenças, desenvolvimento de tecnologias e inovações, atendimento às comunidades indígenas, melhoria na gestão e educação, e implementação de serviços digitais de saúde, respectivamente."
1,"Quais são as principais ações implementadas para enfrentar a pandemia e garantir o direito à saúde no Brasil, conforme descrito no documento?","custeio de leitos de terapia intensiva, a aquisição e distribuição de medicamentos utilizados na intubação orotra - queal de pacientes, o fornecimento de testes para detecção da doença e, em especial, a adoção de estratégia diversificada para compra de vacinas, resultando na maior Campanha de Vacinação da história nacional. Além das ações para enfrentamento da pandemia, o documento registra a continuidade e o aperfeiçoamento das demais ações de saúde pública, garantindo o direito da população brasileira à saúde. São descritos aspectos relevantes do funcionamento do Sistema Único de Saúde (SUS), com informações detalhadas sobre como foram empregados os recursos humanos, físicos e financeiros necessários ao funcionamento do Sistema. O Relatório foi construído coletivamente, com o","As principais ações implementadas para enfrentar a pandemia e garantir o direito à saúde no Brasil incluem o custeio de leitos de terapia intensiva, a aquisição e distribuição de medicamentos para intubação orotra, o fornecimento de testes para detecção da doença e a adoção de uma estratégia diversificada para a compra de vacinas, resultando na maior Campanha de Vacinação da história nacional. Além disso, houve a continuidade e aperfeiçoamento das ações de saúde pública, com informações detalhadas sobre o funcionamento do Sistema Único de Saúde (SUS), incluindo o uso de recursos humanos, físicos e financeiros."
2,"Quais são as principais iniciativas da política de saúde executadas pelo Ministério da Saúde em 2021, conforme o Relatório de Gestão?","envolvimento dos diferentes setores do Ministério, e está organizado em três grandes eixos, para facilitar ao leitor compreender a atuação do Ministério da Saúde na gestão do SUS. Boa leitura a todos! Marcelo Queiroga Ministro da Saúde O RELATÓRIO DE GESTÃO DO MINISTÉRIO DA SAÚDE DISCORRE SOBRE AS PRINCIPAIS INICIATIVAS DA POLÍTICA DE SAÚDE EXECUTADAS EM 2021, TORNANDO PÚBLICOS RESULT TADOS E REFLETINDO SOBRE OBSTÁCULOS A SEREM CONFRONTADOS. VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA014 Segundo o art. 196 da Constituição Federal de 1988, “a saúde é direito de todos e dever do Estado, garantido mediante políticas sociais e econômicas que visem à redução do risco de doença 

## 🤖 Célula 4 — Carregamento do Modelo Fine-Tuned

Carregamos o modelo base `distilgpt2` e aplicamos os adaptadores LoRA salvos em `lora_finetuned_model/`.

> **ℹ️ LoRA (Low-Rank Adaptation):** Técnica de fine-tuning eficiente que treina apenas matrizes de baixo rank adicionadas ao modelo original, reduzindo drasticamente o número de parâmetros treináveis.

In [ ]:
# ============================================================\n",
#  CONFIGURAÇÃO DOS 4 MODELOS ADAPTADOS (LoRA)\n",
# ============================================================\n",

# TODO: Ajuste os nomes das pastas ('lora_folder') conforme aparecem na sua barra lateral do VS Code
CONFIG_MODELOS = {
    "BART": {
        "model_base": "facebook/bart-base", 
        "lora_folder": "./modelos_finais/modelo_final_bart",
        "tokenizer_folder": "./modelos_finais/modelo_final_bart" # ou use o nome base do HF se salvou junto
    },
    "Blenderbot": {
        "model_base": "facebook/blenderbot-400M-distill", 
        "lora_folder": "./modelos_finais/modelo_final_blenderbot",
        "tokenizer_folder": "./modelos_finais/modelo_final_blenderbot"
    },
    "Pythia": {
        "model_base": "EleutherAI/pythia-70m", # Ajuste para a versão exata que você usou (ex: 70m, 160m, 410m, 1b)
        "lora_folder": "./modelos_finais/modelo_final_pythia",
        "tokenizer_folder": "./modelos_finais/modelo_final_pythia"
    },
    "Qwen": {
        "model_base": "Qwen/Qwen2.5-0.5B", # Ajuste para a versão exata que você usou (ex: 0.5B, 1.5B)
        "lora_folder": "./modelos_finais/modelo_final_qwen",
        "tokenizer_folder": "./modelos_finais/modelo_final_qwen"
    }
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Estrutura dos 4 modelos configurada!')
print(f'   Dispositivo que será utilizado: {DEVICE}')

✅ Estrutura dos 4 modelos configurada!
   Dispositivo que será utilizado: cpu


In [5]:
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

def carregar_modelo_e_tokenizer(nome_modelo, info_modelo):
    """
    Carrega dinamicamente o modelo base correto (CausalLM ou Seq2Seq),
    aplica os pesos do LoRA e retorna o modelo junto com seu tokenizer.
    """
    print(f"🔄 Carregando {nome_modelo}...")
    
    # 1. Identifica a arquitetura correta do modelo
    if nome_modelo in ["BART", "Blenderbot"]:
        classe_modelo = AutoModelForSeq2SeqLM
    else: # Pythia e Qwen
        classe_modelo = AutoModelForCausalLM
        
    # 2. Carrega o Tokenizer local
    tokenizer = AutoTokenizer.from_pretrained(info_modelo["tokenizer_folder"])
    
    # 3. Carrega o Modelo Base da internet/cache com as configurações de memória corretas
    base_model = classe_modelo.from_pretrained(
        info_modelo["model_base"],
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    
    # 4. Mescla o modelo base com os seus pesos salvos do LoRA (Fine-Tuned)
    modelo_final = PeftModel.from_pretrained(base_model, info_modelo["lora_folder"])
    modelo_final.to(DEVICE)
    modelo_final.eval() # Coloca em modo de avaliação (desativa dropout)
    
    print(f"✅ {nome_modelo} carregado com sucesso no dispositivo: {DEVICE}\n")
    return modelo_final, tokenizer

## ✍️ Célula 5 — Função de Geração de Respostas

Definimos a função central que constrói o **prompt** e chama o modelo para gerar uma resposta.

In [6]:
# ============================================================
#  FUNÇÃO DE GERAÇÃO DE TEXTO (ADAPTADA PARA 4 MODELOS)
# ============================================================

def build_prompt(instruction: str, input_text: str = '') -> str:
    """
    Constrói o prompt no formato que foi usado no treinamento (LoRA).
    """
    if input_text.strip():
        return (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n'
        )
    return (
        f'### Instruction:\n{instruction}\n\n'
        f'### Response:\n'
    )


def generate_response(
    model,
    tokenizer,
    nome_modelo: str,
    instruction: str,
    input_text: str = '',
    max_new_tokens: int = 200,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Gera a resposta lidando com CausalLM (Pythia, Qwen) e Seq2Seq (BART, Blenderbot).
    """
    # 1. Monta o prompt do mesmo jeito que o modelo aprendeu no treino
    prompt = build_prompt(instruction, input_text)
    
    # 2. Tokenização
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    # Resolve problema de pad_token ausente
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 3. Geração do texto
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 4. Decodificação inteligente baseada na arquitetura
    if nome_modelo in ["BART", "Blenderbot"]:
        # Modelos Seq2Seq retornam APENAS a resposta gerada
        resposta_gerada = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    else:
        # Modelos CausalLM retornam PROMPT + RESPOSTA (precisamos fatiar)
        prompt_len = inputs['input_ids'].shape[1]
        generated_ids = output_ids[0][prompt_len:]
        resposta_gerada = tokenizer.decode(generated_ids, skip_special_tokens=True)
        
    return resposta_gerada.strip()

print('✅ Funções de geração adaptadas para múltiplas arquiteturas definidas!')

✅ Funções de geração adaptadas para múltiplas arquiteturas definidas!


## 📏 Célula 6 — Métrica 1: Perplexidade (PPL)

### O que é?
A **Perplexidade** mede o quão surpreso o modelo fica ao ver o texto de referência. É calculada a partir da **log-verossimilhança negativa média** por token:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i | w_1, \dots, w_{i-1})\right)$$

- **PPL baixa** → o modelo prevê bem o texto de referência → melhor desempenho.
- **PPL = 1** → previsão perfeita (impossível na prática).
- **PPL alta** → o modelo está "confuso" com o texto.

### Como é calculada aqui?
Para cada amostra, passamos o **texto de referência** (output esperado) pelo modelo e calculamos a loss (cross-entropy), depois convertemos para PPL.

In [7]:
# ============================================================
#  MÉTRICA 1 — PERPLEXIDADE (PPL) ADAPTADA (Seq2Seq & CausalLM)
# ============================================================
import math

def compute_perplexity_for_sample(
    model, tokenizer, nome_modelo: str, instruction: str, input_text: str, reference: str
) -> float:
    """
    Calcula a Perplexidade do modelo sobre o texto de referência 
    levando em conta a arquitetura (Seq2Seq ou CausalLM).
    """
    prompt = build_prompt(instruction, input_text)

    # -------------------------------------------------------------
    # Lógica para modelos Seq2Seq (BART, Blenderbot)
    # -------------------------------------------------------------
    if nome_modelo in ["BART", "Blenderbot"]:
        # O Encoder recebe apenas o prompt
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
        # O Decoder recebe apenas a resposta esperada (labels)
        labels = tokenizer(reference, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(input_ids=inputs['input_ids'], labels=labels['input_ids'])
            loss = outputs.loss
            
    # -------------------------------------------------------------
    # Lógica para modelos CausalLM (Pythia, Qwen)
    # -------------------------------------------------------------
    else:
        full_text = prompt + reference
        encodings = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
        
        prompt_tokens = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)['input_ids'].shape[1]
        
        input_ids = encodings['input_ids']
        labels = input_ids.clone()
        # Ignora os tokens do prompt no cálculo do loss (CausalLM)
        labels[0, :prompt_tokens] = -100

        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss

    # Converte o Loss em Perplexidade
    return math.exp(loss.item())

print('✅ Função de Perplexidade atualizada para as duas arquiteturas!')

✅ Função de Perplexidade atualizada para as duas arquiteturas!


## 🔵 Célula 7 — Métrica 2: BLEU

### O que é?
O **BLEU** (Bilingual Evaluation Understudy) mede a **precisão de n-gramas** entre o texto gerado e uma referência.

$$\text{BLEU} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

onde:
- $p_n$ = precisão de n-gramas (1-gram, 2-gram, 3-gram, 4-gram)
- **BP** = Brevity Penalty (penaliza respostas muito curtas)
- Escala de 0 a 100 (ou 0 a 1)

### Limitações
BLEU foca em **precisão** — se todos os n-gramas gerados aparecem na referência. Ele não captura bem paráfrases ou sinônimos.

In [8]:
# ============================================================
#  MÉTRICA 2 — BLEU SCORE (MANTIDA E PREPARADA)
# ============================================================
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

# Garante que o recurso de tokenização do NLTK está baixado
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

def compute_bleu(reference: str, generated: str) -> float:
    """
    Calcula o BLEU score (precisão de n-gramas) entre a referência e o texto gerado.
    Retorna um valor entre 0 e 100.
    """
    ref_tokens = nltk.word_tokenize(reference.lower())
    gen_tokens = nltk.word_tokenize(generated.lower())
    
    # Evita divisão por zero se a geração for vazia
    if not gen_tokens:
        return 0.0
        
    smoothie = SmoothingFunction().method1
    # Multiplicamos por 100 para ficar na escala padrão (0 a 100)
    return sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie) * 100

print('✅ Função de BLEU Score configurada!')

✅ Função de BLEU Score configurada!


## 🟢 Célula 8 — Métrica 3: ROUGE

### O que é?
**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) complementa o BLEU focando no **recall** — quanta informação da referência aparece na resposta gerada.

| Variante | O que mede |
|---|---|
| **ROUGE-1** | Sobreposição de unigramas (palavras individuais) |
| **ROUGE-2** | Sobreposição de bigramas (pares de palavras) |
| **ROUGE-L** | Subsequência Comum mais Longa (LCS) — captura ordem |

Cada métrica retorna **Precisão**, **Recall** e **F1** (harmônica entre os dois).

In [9]:
# ============================================================
#  MÉTRICA 3 — ROUGE SCORE (BASEADO NO ORIGINAL)
# ============================================================
from rouge_score import rouge_scorer

def compute_rouge(reference: str, generated: str) -> dict:
    """
    Calcula as pontuações ROUGE-1, ROUGE-2 e ROUGE-L entre a referência e o texto gerado.
    Baseado estritamente na lógica do notebook original.
    """
    # Inicializa o avaliador oficial do ROUGE com stemmer ativado
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    
    # Extrai o fmeasure (F1-Score) multiplicando por 100 para padronização de escala
    return {
        'rouge1': scores['rouge1'].fmeasure * 100,
        'rouge2': scores['rouge2'].fmeasure * 100,
        'rougeL': scores['rougeL'].fmeasure * 100
    }

print('✅ Função de ROUGE Score configurada com base no código original!')

✅ Função de ROUGE Score configurada com base no código original!


## 🟠 Célula 9 — Métrica 4: Fidelidade (Faithfulness)

### O que é?
A **Fidelidade** avalia se a resposta gerada é **factualmente consistente** com o contexto de entrada (instruction + input). É crítica em sistemas **RAG** (Retrieval-Augmented Generation).

### Como é calculada aqui?
Usamos uma abordagem léxica baseada em **sobreposição de tokens importantes**:
1. Extraímos tokens significativos do contexto (substantivos, números, marcas)
2. Calculamos a fração presente na resposta gerada

> ⚠️ **Nota:** Em produção, Faithfulness é melhor avaliada por um modelo de NLI (Natural Language Inference) como DeBERTa ou via LLM-as-judge.

In [10]:
# ============================================================
#  MÉTRICA 4 — FIDELIDADE (FAITHFULNESS)
# ============================================================

# ============================================================
#  MÉTRICA 4 — FIDELIDADE (FAITHFULNESS) (BASEADO NO ORIGINAL)
# ============================================================
import re

def extract_key_tokens(text: str) -> set:
    """
    Extrai tokens relevantes: números, palavras com maiúscula inicial
    (marcas, modelos), e tokens longos (substantivos específicos).
    Ignora stopwords e tokens muito curtos.
    """
    STOPWORDS = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'could', 'should', 'may', 'might', 'to', 'of', 'in', 'on',
        'at', 'for', 'with', 'by', 'from', 'or', 'and', 'but', 'if',
        'your', 'my', 'our', 'their', 'this', 'that', 'it', 'its',
    }
    tokens = re.findall(r'[A-Za-z0-9]+', text.lower())
    # Mantém números, tokens com >= 4 chars fora de stopwords
    return {
        t for t in tokens
        if (t.isdigit() or len(t) >= 4) and t not in STOPWORDS
    }


def compute_faithfulness(instruction: str, input_text: str, generated: str) -> float:
    """
    Calcula a Fidelidade como a proporção de tokens-chave do contexto
    (instruction + input) que aparece na resposta gerada.

    Score original era 0.0 a 1.0. Multiplicado por 100 para padronização.
    """
    context_tokens   = extract_key_tokens(instruction + ' ' + input_text)
    generated_tokens = extract_key_tokens(generated)

    if not context_tokens:
        return 50.0  # sem contexto mensurável, score neutro (era 0.5, agora 50.0)

    overlap = context_tokens & generated_tokens
    
    # Multiplicamos por 100 para ficar na mesma escala do BLEU e ROUGE na tabela final
    return (len(overlap) / len(context_tokens)) * 100

print('✅ Função de Fidelidade configurada com base no código original!')

✅ Função de Fidelidade configurada com base no código original!


## 🟡 Célula 10 — Métrica 5: Relevância da Resposta (Answer Relevance)

### O que é?
A **Relevância** mede se a resposta realmente **responde à pergunta feita**. Uma resposta pode ser fluente e fiel ao contexto, mas não responder à instrução específica.

### Como é calculada aqui?
Usamos **Jaccard Similarity** sobre tokens-chave entre a instrução e a resposta gerada:

$$\text{Relevância} = \frac{|\text{tokens}_{\text{instrução}} \cap \text{tokens}_{\text{resposta}}|}{|\text{tokens}_{\text{instrução}} \cup \text{tokens}_{\text{resposta}}|}$$

> ⚠️ Em produção, use embeddings semânticos (sentence-transformers) ou LLM-as-judge para maior precisão.

In [11]:
# ============================================================
#  MÉTRICA 5 — RELEVÂNCIA DA RESPOSTA (ANSWER RELEVANCE)
# ============================================================

def compute_answer_relevance(instruction: str, generated: str) -> float:
    """
    Calcula a Relevância como Jaccard Similarity entre os tokens-chave
    da instrução e da resposta gerada.

    Score original era 0.0 a 1.0. Multiplicado por 100 para padronização.
    """
    # Reutiliza a função extract_key_tokens definida na célula de Fidelidade (Célula 9)
    instr_tokens = extract_key_tokens(instruction)
    gen_tokens   = extract_key_tokens(generated)

    if not instr_tokens and not gen_tokens:
        return 50.0  # Sem tokens mensuráveis, score neutro (era 0.5, agora 50.0)

    intersection = instr_tokens & gen_tokens
    union        = instr_tokens | gen_tokens

    # Jaccard puro penaliza respostas muito longas — adicionamos bônus
    # de cobertura da instrução para valorizar completude
    jaccard  = len(intersection) / len(union)
    coverage = len(intersection) / len(instr_tokens) if instr_tokens else 0

    # Multiplicamos por 100 para padronizar a escala de 0 a 100 na tabela final
    return ((jaccard + coverage) / 2.0) * 100

print('✅ Função de Relevância configurada com base no código original!')

✅ Função de Relevância configurada com base no código original!


## 🔴 Célula 11 — Métrica 6: Aderência ao Plano (Plan Adherence)

### O que é?
A **Aderência ao Plano** mede se o agente de IA segue a estrutura esperada de resposta — relevante em contextos de **AI Agents** que devem seguir planos de ação predefinidos.

### Como é calculada aqui?
Para nosso dataset automotivo, o "plano" esperado é a estrutura do output de referência:
- ✅ Numeração de passos (`1. 2. 3.`)
- ✅ Uso de bullet points (`-`, `•`)
- ✅ Formato de valores técnicos (ex.: `35 psi`, `10,000 miles`)
- ✅ Presença de seções com dois-pontos

Avaliamos se a resposta gerada segue os mesmos **elementos estruturais** que a referência.

In [12]:
# ============================================================
#  MÉTRICA 6 — ADERÊNCIA AO PLANO (PLAN ADHERENCE)
# ============================================================
import re
import numpy as np

def detect_structural_elements(text: str) -> dict:
    """
    Detecta elementos estruturais no texto:
    - has_numbered_list : tem lista numerada (1. 2. ...)
    - has_bullets       : tem bullet points (- ou •)
    - has_technical     : tem valores técnicos (números + unidade)
    - has_sections      : tem seções com dois-pontos no início de linha
    - step_count        : número de passos numerados
    """
    return {
        'has_numbered_list': bool(re.search(r'^\d+\.\s', text, re.MULTILINE)),
        'has_bullets'      : bool(re.search(r'^[\-•]\s', text, re.MULTILINE)),
        'has_technical'    : bool(re.search(
            r'\d+[,.]?\d*\s*(psi|mph|lbs|miles|km|liter|L|V\d|octane|months?)',
            text, re.IGNORECASE
        )),
        'has_sections'     : bool(re.search(r'^[A-Z][^\n]+:\s*$', text, re.MULTILINE)),
        'step_count'       : len(re.findall(r'^\d+\.\s', text, re.MULTILINE)),
    }

def compute_plan_adherence(reference: str, generated: str) -> float:
    """
    Calcula a Aderência ao Plano comparando os elementos estruturais
    do texto de referência com os da resposta gerada.

    Score original era 0.0 a 1.0. Multiplicado por 100 para padronização.
    """
    ref_struct = detect_structural_elements(reference)
    gen_struct = detect_structural_elements(generated)

    checks = []
    binary_features = ['has_numbered_list', 'has_bullets', 'has_technical', 'has_sections']

    for feat in binary_features:
        if ref_struct[feat]:  # Só avalia se a referência tem esse elemento
            checks.append(1.0 if gen_struct[feat] else 0.0)

    # Verifica se contagem de passos é similar (±1)
    if ref_struct['step_count'] > 0:
        step_ratio = min(gen_struct['step_count'], ref_struct['step_count']) / \
                     max(gen_struct['step_count'], ref_struct['step_count'], 1)
        checks.append(step_ratio)

    # Retorna 50.0 (neutro) se não houver checks, senão calcula a média e multiplica por 100
    if not checks:
        return 50.0
        
    return np.mean(checks) * 100

print('✅ Função de Aderência ao Plano configurada com base no código original!')

✅ Função de Aderência ao Plano configurada com base no código original!


## 📋 Célula 12 — Tabela Consolidada de Resultados

Unificamos todos os scores por amostra em um único DataFrame para análise comparativa.

In [ ]:
# ============================================================
#  CÉLULA 12 — TABELA CONSOLIDADA DETALHADA (4 MODELOS) + EXPORT RESUMO
# ============================================================
import json
import pandas as pd
import gc
import torch
import numpy as np

# 1. Carrega o seu dataset de testes (dataset.jsonl)
dataset_path = "dataset.jsonl"
samples = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f"📋 Dataset carregado com {len(samples)} amostras.\n")

# Dicionário para guardar as tabelas estilizadas de cada modelo
tabelas_modelos = {}

# 2. O Grande Loop: Avalia e cria a tabela para cada um dos 4 modelos
for nome_modelo, info_modelo in CONFIG_MODELOS.items():
    print(f"🔄 Gerando relatório detalhado para o modelo: {nome_modelo}...")
    
    # Carrega o modelo da vez e limpa os acumuladores
    model, tokenizer = carregar_modelo_e_tokenizer(nome_modelo, info_modelo)
    
    ppl_scores = []
    bleu_scores = []
    rouge1_scores, rouge2_scores, l_scores = [], [], []
    faithfulness_scores = []
    relevance_scores = []
    plan_scores = []
    
    # Avalia amostra por amostra (Linhas 0, 1, 2... da sua tabela)
    for s in samples:
        instruction = s.get("instruction", s.get("pergunta", ""))
        input_text = s.get("input", s.get("contexto", ""))
        reference = s.get("output", s.get("resposta_referencia", ""))
        
        # Gera a resposta purificada
        gen = generate_response(model, tokenizer, nome_modelo, instruction, input_text)
        
        # Calcula a Perplexidade
        try:
            ppl = compute_perplexity_for_sample(model, tokenizer, nome_modelo, instruction, input_text, reference)
            ppl_scores.append(ppl if ppl != float('inf') and not np.isnan(ppl) else 100.0)
        except:
            ppl_scores.append(100.0)
            
        # Calcula as demais métricas baseadas no seu formato atualizado (0-100)
        bleu_scores.append(compute_bleu(reference, gen))
        
        r_sc = compute_rouge(reference, gen)
        rouge1_scores.append(r_sc['rouge1'])
        rouge2_scores.append(r_sc['rouge2'])
        l_scores.append(r_sc['rougeL'])
        
        faithfulness_scores.append(compute_faithfulness(instruction, input_text, gen))
        relevance_scores.append(compute_answer_relevance(instruction, gen))
        plan_scores.append(compute_plan_adherence(reference, gen))
        
    # 3. Monta a estrutura da tabela de amostras igual à sua original
    df_results = pd.DataFrame({
        'Instrução'           : [s.get('instruction', s.get('pergunta', ''))[:50] + '...' for s in samples],
        'PPL'                 : [round(p, 2) for p in ppl_scores],
        'BLEU'                : [round(b, 2) for b in bleu_scores],
        'ROUGE-1 F1'          : [round(r, 2) for r in rouge1_scores],
        'ROUGE-2 F1'          : [round(r, 2) for r in rouge2_scores],
        'ROUGE-L F1'          : [round(r, 2) for r in l_scores],
        'Faithfulness'        : [round(f, 2) for f in faithfulness_scores],
        'Answer Relevance'    : [round(r, 2) for r in relevance_scores],
        'Plan Adherence'      : [round(p, 2) for p in plan_scores],
    })

    # 4. Cria a linha consolidada de médias (Idêntica à sua linha 10 da imagem)
    means_row = pd.DataFrame([{
        'Instrução'        : '📊 MÉDIA',
        'PPL'              : round(np.mean(ppl_scores), 2),
        'BLEU'             : round(np.mean(bleu_scores), 2),
        'ROUGE-1 F1'       : round(np.mean(rouge1_scores), 2),
        'ROUGE-2 F1'       : round(np.mean(rouge2_scores), 2),
        'ROUGE-L F1'       : round(np.mean(l_scores), 2),
        'Faithfulness'     : round(np.mean(faithfulness_scores), 2),
        'Answer Relevance' : round(np.mean(relevance_scores), 2),
        'Plan Adherence'   : round(np.mean(plan_scores), 2),
    }])

    df_display = pd.concat([df_results, means_row], ignore_index=True)
    
    # 5. Aplica a sua função original de estilização visual
    def highlight_mean_row(row):
        if row['Instrução'] == '📊 MÉDIA':
            return ['background-color: #1a1a2e; color: #e0e0e0; font-weight: bold'] * len(row)
        return [''] * len(row)

    # Aplica os gradientes (ajustado para vmax=100 já que as notas agora vão até 100)
    styled = (
        df_display.style
        .apply(highlight_mean_row, axis=1)
        .background_gradient(subset=['ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1',
                                      'Faithfulness', 'Answer Relevance', 'Plan Adherence'],
                             cmap='YlGn', vmin=0, vmax=100)
        .background_gradient(subset=['PPL'], cmap='YlOrRd_r')
        .set_caption(f'📋 Resultados por Amostra — Modelo: {nome_modelo}')
    )
    
    # Salva a tabela estilizada deste modelo na nossa coleção
    tabelas_modelos[nome_modelo] = styled
    
    # 6. FAXINA DE MEMÓRIA (O segredo para não travar o seu VS Code)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"🧹 Dados processados e memória limpa para {nome_modelo}.\n")


# ============================================================
# 7. EXTRAÇÃO E EXPORTAÇÃO AUTOMÁTICA DO RESUMO PARA A ÚLTIMA CÉLULA
# ============================================================
print("💾 Gerando arquivo de resumos para o Relatório Final...")
resumo_medias = []

for nome_modelo, tabela_estilizada in tabelas_modelos.items():
    # Recupera o DataFrame limpo de dentro do objeto Styler do pandas
    df_original = tabela_estilizada.data
    
    # Extrai a linha que criamos com o nome '📊 MÉDIA'
    linha_media = df_original[df_original['Instrução'] == '📊 MÉDIA'].copy()
    linha_media['Modelo'] = nome_modelo
    resumo_medias.append(linha_media)

# Consolida as 4 linhas de médias em uma estrutura única global
df_tabela_consolidada = pd.concat(resumo_medias, ignore_index=True)

# Organiza as colunas exatamente no formato esperado pelo relatório técnico
colunas_ordenadas = ['Modelo', 'PPL', 'BLEU', 'ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1', 'Faithfulness', 'Answer Relevance', 'Plan Adherence']
df_tabela_consolidada = df_tabela_consolidada[colunas_ordenadas]

# Salva fisicamente o CSV no diretório para garantir compatibilidade
df_tabela_consolidada.to_csv("tabela_consolidada_resultados.csv", index=False, encoding='utf-8-sig')
print("✅ Arquivo 'tabela_consolidada_resultados.csv' criado e variável 'df_tabela_consolidada' pronta na memória!\n")


# ============================================================
# 8. EXIBIÇÃO VISUAL DAS 4 TABELAS COLORIDAS NO JUPYTER
# ============================================================
print("="*60)
print("📊 EXIBINDO AS TABELAS CONSOLIDADAS DOS 4 MODELOS")
print("="*60)

for nome_modelo, tabela_colorida in tabelas_modelos.items():
    display(tabela_colorida)    

📋 Dataset carregado com 106 amostras.

🔄 Gerando relatório detalhado para o modelo: BART...
🔄 Carregando BART...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 259/259 [00:00<00:00, 9582.54it/s]


✅ BART carregado com sucesso no dispositivo: cpu



KeyboardInterrupt: 

## 📈 Célula 13 — Visualizações

Gráficos que facilitam a interpretação dos resultados.

In [ ]:
# ============================================================
#  CÉLULA 13 — PAINEL DE VISUALIZAÇÕES (PARA CADA MODELO)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch

# A) Função baseada estritamente no seu código original
def plot_model_dashboard(nome_modelo, ppl_scores, bleu_scores, rouge1, rouge2, rougel, faith, relev, plan, n_samples):
    """
    Gera o painel de 6 gráficos idêntico ao original, adaptado para a escala 0-100
    e salvando o arquivo com o nome do modelo.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'📊 Avaliação do Modelo — {nome_modelo}', fontsize=18, fontweight='bold', y=1.02)

    sample_labels = [f'S{i+1}' for i in range(n_samples)]
    colors = plt.cm.Set2.colors

    # --- 1. Perplexidade por Amostra ---
    ax = axes[0, 0]
    bars = ax.bar(sample_labels, ppl_scores, color=colors[0], edgecolor='white', linewidth=0.5)
    ax.axhline(np.mean(ppl_scores), color='red', linestyle='--', linewidth=1.5, label=f'Média: {np.mean(ppl_scores):.1f}')
    ax.set_title('Perplexidade (PPL)', fontweight='bold')
    ax.set_ylabel('PPL (menor = melhor)')
    ax.set_xlabel('Amostra')
    ax.legend(fontsize=9)

    # --- 2. BLEU por Amostra ---
    ax = axes[0, 1]
    ax.bar(sample_labels, bleu_scores, color=colors[1], edgecolor='white', linewidth=0.5)
    ax.axhline(np.mean(bleu_scores), color='red', linestyle='--', linewidth=1.5,
               label=f'Média: {np.mean(bleu_scores):.1f}')
    ax.set_title('BLEU Score por Amostra', fontweight='bold')
    ax.set_ylabel('BLEU (0–100)')
    ax.set_xlabel('Amostra')
    ax.legend(fontsize=9)

    # --- 3. ROUGE F1 Comparativo ---
    ax = axes[0, 2]
    x = np.arange(len(sample_labels))
    w = 0.28
    ax.bar(x - w, rouge1, w, label='ROUGE-1', color=colors[2], edgecolor='white')
    ax.bar(x,     rouge2, w, label='ROUGE-2', color=colors[3], edgecolor='white')
    ax.bar(x + w, rougel, w, label='ROUGE-L', color=colors[4], edgecolor='white')
    ax.set_title('ROUGE F1 por Amostra', fontweight='bold')
    ax.set_ylabel('F1 Score (0–100)')
    ax.set_xlabel('Amostra')
    ax.set_xticks(x)
    ax.set_xticklabels(sample_labels)
    ax.legend(fontsize=9)

    # --- 4. Faithfulness, Relevance, Plan Adherence ---
    ax = axes[1, 0]
    ax.plot(sample_labels, faith, 'o-', color=colors[0], label='Faithfulness', linewidth=2)
    ax.plot(sample_labels, relev, 's-', color=colors[1], label='Answer Relevance', linewidth=2)
    ax.plot(sample_labels, plan, '^-', color=colors[2], label='Plan Adherence', linewidth=2)
    ax.set_ylim(0, 105) # Ajustado para escala 100
    ax.set_title('Métricas de Qualidade por Amostra', fontweight='bold')
    ax.set_ylabel('Score (0–100)')
    ax.set_xlabel('Amostra')
    ax.legend(fontsize=9)

    # --- 5. Radar / Spider Chart das Médias ---
    ax = axes[1, 1]
    ax.remove()
    ax_radar = fig.add_subplot(2, 3, 5, projection='polar')

    metric_names  = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Faithful', 'Relevance', 'Plan']
    metric_values = [
        np.mean(rouge1), np.mean(rouge2), np.mean(rougel),
        np.mean(faith), np.mean(relev), np.mean(plan),
    ]

    angles = np.linspace(0, 2 * np.pi, len(metric_names), endpoint=False).tolist()
    values_closed = metric_values + [metric_values[0]]
    angles_closed = angles + [angles[0]]

    ax_radar.plot(angles_closed, values_closed, 'o-', linewidth=2, color=colors[3])
    ax_radar.fill(angles_closed, values_closed, alpha=0.25, color=colors[3])
    ax_radar.set_xticks(angles)
    ax_radar.set_xticklabels(metric_names, size=9)
    ax_radar.set_ylim(0, 100) # Ajustado para escala 100
    ax_radar.set_title('Radar das Médias (0–100)', fontweight='bold', pad=15)

    # --- 6. Sumário em barras horizontais ---
    ax = axes[1, 2]
    summary_names = [
        'ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1',
        'Faithfulness', 'Answer Relevance', 'Plan Adherence'
    ]
    summary_vals = metric_values
    bar_colors   = [colors[i % len(colors)] for i in range(len(summary_names))]
    h_bars = ax.barh(summary_names, summary_vals, color=bar_colors, edgecolor='white')
    ax.set_xlim(0, 100) # Ajustado para escala 100
    ax.set_title('Resumo das Métricas (Médias)', fontweight='bold')
    ax.set_xlabel('Score médio (0-100)')
    
    for bar, val in zip(h_bars, summary_vals):
        ax.text(val + 1.0, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', fontsize=9)

    plt.tight_layout()
    
    # Salva o arquivo PNG com o nome do modelo para não sobrescrever
    filename = f'avaliacao_metricas_{nome_modelo.replace("/", "_")}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Figura do modelo {nome_modelo} salva como {filename}\n')


# ============================================================
# B) EXECUÇÃO: Gerando dados e chamando os gráficos para os 4 Modelos
# ============================================================
print("🔄 Preparando geração de gráficos para todos os modelos...\n")

for nome_modelo, info_modelo in CONFIG_MODELOS.items():
    print(f"📊 Coletando dados para o gráfico de: {nome_modelo}")
    
    model, tokenizer = carregar_modelo_e_tokenizer(nome_modelo, info_modelo)
    
    m_ppl, m_bleu = [], []
    m_r1, m_r2, m_rl = [], [], []
    m_faith, m_rel, m_plan = [], [], []
    
    for s in samples:
        inst = s.get("instruction", s.get("pergunta", ""))
        inp = s.get("input", s.get("contexto", ""))
        ref = s.get("output", s.get("resposta_referencia", ""))
        
        gen = generate_response(model, tokenizer, nome_modelo, inst, inp)
        
        try:
            p = compute_perplexity_for_sample(model, tokenizer, nome_modelo, inst, inp, ref)
            m_ppl.append(p if p != float('inf') and not np.isnan(p) else 100.0)
        except:
            m_ppl.append(100.0)
            
        m_bleu.append(compute_bleu(ref, gen))
        r_sc = compute_rouge(ref, gen)
        m_r1.append(r_sc['rouge1']); m_r2.append(r_sc['rouge2']); m_rl.append(r_sc['rougeL'])
        m_faith.append(compute_faithfulness(inst, inp, gen))
        m_rel.append(compute_answer_relevance(inst, gen))
        m_plan.append(compute_plan_adherence(ref, gen))

    # Chama a sua função de plotagem para este modelo
    plot_model_dashboard(
        nome_modelo, m_ppl, m_bleu, 
        m_r1, m_r2, m_rl, 
        m_faith, m_rel, m_plan, len(samples)
    )
    
    # Faxina!
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 🔬 Célula 14 — Análise Qualitativa: Comparação Resposta vs. Referência

Examinamos lado a lado as respostas geradas vs. as esperadas para 3 amostras selecionadas.

In [ ]:
# ============================================================
#  CÉLULA 14 — ANÁLISE QUALITATIVA (4 MODELOS)
# ============================================================
import json
import gc
import torch
import numpy as np

# 1. Carrega o dataset
dataset_path = "dataset.jsonl"
samples = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

INSPECT_INDICES = [0, 3, 6]  # Altere para ver outras amostras

# 2. Roda a inspeção para cada modelo
for nome_modelo, info_modelo in CONFIG_MODELOS.items():
    print("\n\n" + "#" * 70)
    print(f"#  ANÁLISE QUALITATIVA DO MODELO: {nome_modelo}")
    print("#" * 70)

    # Carrega o modelo da vez
    model, tokenizer = carregar_modelo_e_tokenizer(nome_modelo, info_modelo)

    for idx in INSPECT_INDICES:
        # Evita erro caso o índice escolhido seja maior que o dataset
        if idx >= len(samples):
            continue
            
        s = samples[idx]
        instruction = s.get("instruction", s.get("pergunta", ""))
        input_text = s.get("input", s.get("contexto", ""))
        reference = s.get("output", s.get("resposta_referencia", ""))

        # Gera a resposta para esta amostra
        gen = generate_response(model, tokenizer, nome_modelo, instruction, input_text)

        # Calcula as métricas especificamente para esta geração
        try:
            ppl = compute_perplexity_for_sample(model, tokenizer, nome_modelo, instruction, input_text, reference)
            ppl = ppl if (ppl != float('inf') and not np.isnan(ppl)) else 100.0
        except:
            ppl = 100.0
            
        bleu = compute_bleu(reference, gen)
        r_sc = compute_rouge(reference, gen)
        rouge_l = r_sc['rougeL']
        faith = compute_faithfulness(instruction, input_text, gen)
        relev = compute_answer_relevance(instruction, gen)
        plan = compute_plan_adherence(reference, gen)

        # Print visual idêntico ao seu original
        print('\n' + '=' * 70)
        print(f'🔎 AMOSTRA {idx + 1}')
        print(f'📌 Instrução : {instruction}')
        if input_text:
            print(f'📥 Contexto  : {input_text}')
        print()
        print('✅ REFERÊNCIA:')
        print(reference)
        print()
        print('🤖 GERADO:')
        print(gen if gen else '[sem saída gerada]')
        print()
        print(f'   PPL              = {ppl:.2f}')
        print(f'   BLEU             = {bleu:.2f}')
        print(f'   ROUGE-L F1       = {rouge_l:.2f}')
        print(f'   Faithfulness     = {faith:.2f}')
        print(f'   Answer Relevance = {relev:.2f}')
        print(f'   Plan Adherence   = {plan:.2f}')
        print()

    # Faxina de memória
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 📝 Célula 15 — Relatório Final

Geração automática de um relatório de texto com interpretação dos resultados.

In [ ]:
# ============================================================
#  CÉLULA 15 — RELATÓRIO FINAL AUTOMÁTICO (COMPARATIVO)
# ============================================================
import pandas as pd
import numpy as np
import os

def interpret(metric: str, value: float) -> str:
    """Interpreta as métricas na escala 0 a 100."""
    if pd.isna(value) or value == "N/A":
        return '⚪ Inválido/Erro'
        
    value = float(value)
    
    # Para PPL, menor é melhor
    if metric == 'ppl':
        if value <= 20: return '🟢 Bom'
        elif value <= 60: return '🟡 Moderado'
        else: return '🔴 Fraco'
        
    thresholds = {
        'rouge'       : [(50, '🟢 Bom'), (20, '🟡 Moderado'), (0, '🔴 Fraco')],
        'faith_rel_pl': [(70, '🟢 Bom'), (40, '🟡 Moderado'), (0, '🔴 Fraco')],
        'bleu'        : [(20, '🟢 Bom'), (5,  '🟡 Moderado'), (0, '🔴 Fraco')],
    }
    
    key = 'bleu' if metric == 'bleu' else ('rouge' if 'rouge' in metric else 'faith_rel_pl')
    
    for threshold, label in thresholds[key]:
        if value >= threshold:
            return label
    return '🔴 Fraco'

# 1. Tenta carregar os resultados da Célula 12 (Via CSV ou Memória)
csv_path = "tabela_consolidada_resultados.csv"
try:
    if os.path.exists(csv_path):
        df_res = pd.read_csv(csv_path)
    else:
        df_res = df_tabela_consolidada # Tenta pegar da memória se o CSV não existir
except NameError:
    print("❌ ERRO: Rode a Célula 12 primeiro para gerar os resultados consolidados!")
    df_res = None

# 2. Gera o relatório se os dados existirem
if df_res is not None:
    print('=' * 75)
    print('         📋  RELATÓRIO DE AVALIAÇÃO COMPARATIVA — 4 MODELOS')
    print('=' * 75)
    print(f'  Dataset Utilizado : dataset.jsonl')
    print(f'  Escala das Notas  : 0 a 100 (Exceto PPL)')
    print('=' * 75)
    
    melhor_modelo_rag = ""
    maior_score_rag = -1
    
    # Loop para imprimir o boletim de cada modelo
    for index, row in df_res.iterrows():
        modelo = row['Modelo']
        ppl    = row.get('Perplexidade (PPL) ↓', 100.0)
        bleu   = row.get('BLEU ↑', 0.0)
        r1     = row.get('ROUGE-1 ↑', 0.0)
        r2     = row.get('ROUGE-2 ↑', 0.0)
        rL     = row.get('ROUGE-L ↑', 0.0)
        faith  = row.get('Fidelidade ↑', 0.0)
        relev  = row.get('Relevância ↑', 0.0)
        plan   = row.get('Aderência ao Plano ↑', 0.0)
        
        # Lógica para achar o melhor modelo para RAG/Agentes (Média de Fidelidade + Aderência)
        score_rag = (float(faith) + float(plan)) / 2
        if score_rag > maior_score_rag:
            maior_score_rag = score_rag
            melhor_modelo_rag = modelo

        print(f'\n🤖 MODELO: {modelo}')
        print('-' * 75)
        print(f'  PERPLEXIDADE (PPL)')
        print(f'    Média : {ppl}  {interpret("ppl", ppl)}')
        print()
        print(f'  SOBREPOSIÇÃO LÉXICA (Tradução/Semelhança exata)')
        print(f'    BLEU       : {bleu}  {interpret("bleu", bleu)}')
        print(f'    ROUGE-1 F1 : {r1}  {interpret("rouge1", r1)}')
        print(f'    ROUGE-2 F1 : {r2}  {interpret("rouge2", r2)}')
        print(f'    ROUGE-L F1 : {rL}  {interpret("rougeL", rL)}')
        print()
        print(f'  QUALIDADE PARA RAG E AGENTES (Heurísticas)')
        print(f'    Fidelidade (Faithfulness)   : {faith}  {interpret("faith", faith)}')
        print(f'    Relevância (Ans. Relevance) : {relev}  {interpret("relev", relev)}')
        print(f'    Aderência (Plan Adherence)  : {plan}  {interpret("plan", plan)}')
        print('=' * 75)

    # Conclusão Global Dinâmica
    print('\n' + '─' * 75)
    print('  💡 ANÁLISE GLOBAL AUTOMÁTICA:')
    print(f'     Ao avaliar os 4 modelos testados, observamos diferentes perfis de fluência')
    print(f'     e capacidade de seguir instruções estruturadas.')
    print(f'     Para pipelines de RAG (Geração Aumentada por Recuperação) e Agentes,')
    print(f'     as métricas de Fidelidade e Aderência ao Plano são as mais críticas.')
    print(f'     🏆 Com base nisso, o modelo mais adequado parece ser o: **{melhor_modelo_rag}**.')
    print('─' * 75)

## 💾 Célula 16 — Exportar Resultados

Salvamos os resultados detalhados em CSV e o gráfico em PNG.

In [ ]:
# ============================================================
#  CÉLULA 16 — EXPORTAÇÃO DE RESULTADOS DETALHADOS (MASTER CSV)
# ============================================================
import json
import pandas as pd
import gc
import torch
import numpy as np

# 1. Carrega o dataset de testes
dataset_path = "dataset.jsonl"
samples = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

# Lista que vai acumular os dados completos de todos os modelos
dados_detalhados_completos = []

print("💾 Iniciando a consolidação e exportação dos dados detalhados...\n")

# 2. Loop para extrair textos e métricas por amostra de cada modelo
for nome_modelo, info_modelo in CONFIG_MODELOS.items():
    print(f"📦 Compilando textos e métricas para: {nome_modelo}")
    
    model, tokenizer = carregar_modelo_e_tokenizer(nome_modelo, info_modelo)
    
    for idx, s in enumerate(samples):
        instruction = s.get("instruction", s.get("pergunta", ""))
        input_text = s.get("input", s.get("contexto", ""))
        reference = s.get("output", s.get("resposta_referencia", ""))
        
        # Gera a resposta textual completa
        gen = generate_response(model, tokenizer, nome_modelo, instruction, input_text)
        
        # Calcula as métricas individuais da linha
        try:
            ppl = compute_perplexity_for_sample(model, tokenizer, nome_modelo, instruction, input_text, reference)
            ppl = ppl if (ppl != float('inf') and not np.isnan(ppl)) else 100.0
        except:
            ppl = 100.0
            
        bleu = compute_bleu(reference, gen)
        r_sc = compute_rouge(reference, gen)
        faith = compute_faithfulness(instruction, input_text, gen)
        relev = compute_answer_relevance(instruction, gen)
        plan = compute_plan_adherence(reference, gen)
        
        # Cria uma linha ultra detalhada combinando textos e scores
        dados_detalhados_completos.append({
            'Modelo'          : nome_modelo,
            'Amostra'         : f'S{idx + 1}',
            'Instrução'       : instruction,
            'Contexto'        : input_text if input_text else '[Sem contexto]',
            'Referência'      : reference,
            'Gerado'          : gen if gen else '[Sem saída gerada]',
            'PPL'             : round(ppl, 2),
            'BLEU'            : round(bleu, 2),
            'ROUGE-1 F1'      : round(r_sc['rouge1'], 2),
            'ROUGE-2 F1'      : round(r_sc['rouge2'], 2),
            'ROUGE-L F1'      : round(r_sc['rougeL'], 2),
            'Faithfulness'    : round(faith, 2),
            'Answer Relevance': round(relev, 2),
            'Plan Adherence'  : round(plan, 2)
        })
        
    # Limpeza preventiva de memória de cada modelo
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 3. Transforma tudo em DataFrame e exporta para o arquivo final
df_master_export = pd.DataFrame(dados_detalhados_completos)
CSV_PATH = 'resultados_detalhados_modelos.csv'
df_master_export.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

# 4. Print de sucesso idêntico ao seu layout visual original
print('\n' + '=' * 75)
print('✅ Resultados exportados com sucesso!')
print(f'   📄 {CSV_PATH}       — Métricas por amostra + textos completos (4 modelos)')
print(f'   🖼️  avaliacao_metricas_*.png   — Gráficos de visão geral salvos no diretório')
print('=' * 75)
print('🎉 Avaliação concluída! Todos os artefatos foram gerados com sucesso.')

---

## 📚 Referências e Próximos Passos

### Referências
| Recurso | Link |
|---|---|
| BLEU original | Papineni et al., 2002 |
| ROUGE original | Lin, 2004 |
| Perplexidade | Jelinek et al., 1977 |
| RAGAs (Faithfulness/Relevance) | [github.com/explodinggradients/ragas](https://github.com/explodinggradients/ragas) |
| LoRA | Hu et al., 2021 |

### Próximos Passos Sugeridos

1. **Faithfulness com NLI**: substituir a abordagem léxica por um modelo DeBERTa treinado em NLI.
2. **LLM-as-judge**: usar GPT-4 ou Claude para avaliar Relevância e Plan Adherence de forma semântica.
3. **RAGAs**: integrar o framework [RAGAs](https://github.com/explodinggradients/ragas) para avaliação automatizada de pipelines RAG.
4. **Comparação baseline**: avaliar o `distilgpt2` sem LoRA e comparar com o fine-tuned para quantificar o ganho.
5. **Human Evaluation**: coletar avaliações humanas para validar as métricas automáticas.